# Fehlende Werte ergänzen: Imputation ohne Data Leakage

Dieses Beispiel zeigt, wie du fehlende Werte in Merkmalen mit dem Median ersetzt. Entscheidend: Der Imputer wird **nur mit den Trainingsdaten angepasst**. Anschliessend verwendet er dieselben Trainingsmediane für Training, Validation und Test.

Wir verwenden 10 Beobachtungen und 4 numerische Features. Für die kleine Demonstration teilen wir sie sichtbar in 6 Trainings-, 2 Validierungs- und 2 Testbeobachtungen auf. Bei echten Daten solltest du die Aufteilung normalerweise zufällig vornehmen.

Benötigt: `pandas`, `scikit-learn`.

In [ ]:
import pandas as pd
from sklearn.impute import SimpleImputer

data = pd.DataFrame({
    "Alter":     [20, 30, 40, 50, 60, 70, 80, 55, 35, None],
    "Einkommen": [30, None, 50, 60, 70, 80, 90, 100, 110, 120],
    "Zimmer":    [2, 3, 4, 5, None, 7, 8, 6, 3, 4],
    "Distanz":   [1, 2, 3, 4, 5, 6, 7, 5, 2, 3],
}, index=range(1, 11))

data.index.name = "ID"
data

In diesem Beispiel ist `None` ein fehlender Wert. Pandas zeigt solche Werte als `NaN` an. Es gibt insgesamt drei fehlende Featurewerte: Einkommen und Zimmer im Training sowie Alter im Test.

In [ ]:
X_train = data.loc[1:6]
X_val = data.loc[7:8]
X_test = data.loc[9:10]

print("Fehlende Werte im Training:")
display(X_train.isna().sum())
print("Fehlende Werte in Validation und Test:")
display(pd.DataFrame({
    "Validation": X_val.isna().sum(),
    "Test": X_test.isna().sum(),
}))

## 1. Imputer am Training anpassen

`fit` berechnet für jedes Feature den Median aus den vorhandenen **Trainingswerten**. Diese Mediane kannst du mit `statistics_` ansehen. Der Imputer sieht dabei weder Validation noch Test.

In [ ]:
imputer = SimpleImputer(strategy="median")
imputer.fit(X_train)

train_medians = pd.Series(imputer.statistics_, index=X_train.columns, name="Trainingsmedian")
train_medians

Die erwarteten Trainingsmediane sind Alter `45`, Einkommen `60`, Zimmer `4` und Distanz `3.5`. Zum Beispiel ist der Median des Einkommens nicht aus der gesamten Tabelle berechnet, sondern nur aus den nicht fehlenden Trainingswerten `30, 50, 60, 70, 80`.

## 2. Dieselben Mediane überall anwenden

`transform` ersetzt die fehlenden Werte mit den bereits gelernten Trainingsmedianen. Die folgende kleine Hilfsfunktion erhält beim Umwandeln die Spaltennamen und Zeilen-IDs.

In [ ]:
def impute_frame(frame):
    """Apply the fitted imputer while preserving the DataFrame labels."""
    return pd.DataFrame(
        imputer.transform(frame),
        columns=frame.columns,
        index=frame.index,
    )


X_train_clean = impute_frame(X_train)
X_val_clean = impute_frame(X_val)
X_test_clean = impute_frame(X_test)

print("Trainingsdaten nach der Imputation:")
display(X_train_clean)
print("Testdaten nach der Imputation:")
display(X_test_clean)

Das fehlende Einkommen in ID 2 wird mit `60` gefüllt, das fehlende Zimmer in ID 5 mit `4` und das fehlende Alter in der Test-ID 10 mit dem Trainingsmedian `45`. Der Testwert hat also **nicht** dazu beigetragen, den Median zu berechnen.

In [ ]:
assert X_train_clean.isna().sum().sum() == 0
assert X_val_clean.isna().sum().sum() == 0
assert X_test_clean.isna().sum().sum() == 0
assert X_test_clean.loc[10, "Alter"] == train_medians["Alter"]
print("Alle fehlenden Werte wurden mit den Trainingsmedianen ergänzt.")

## 3. Warum nicht vor dem Split auffüllen?

Wenn du den Median vor der Aufteilung aus allen Daten berechnest, fliessen Informationen aus Validation und Test in die Vorverarbeitung ein. Das ist **Data Leakage** und kann die spätere Leistungsbewertung zu optimistisch machen.

Für Cross-Validation oder den normalen Modellablauf ist eine Pipeline praktisch: Bei jedem Trainingslauf wird der Imputer nur auf dem jeweiligen Trainingsteil angepasst und danach auf die zugehörigen Validierungsdaten angewendet.

```python
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline

pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", LinearRegression()),
])
pipeline.fit(X_train, y_train)
vorhersagen = pipeline.predict(X_test)
```

Die feste Aufteilung der 10 Beispielzeilen ist nur zur Veranschaulichung. Bei einem echten Projekt teilst du die Daten zuerst passend auf (zum Beispiel zufällig oder bei Zeitreihen chronologisch), und passt dann Imputer, Scaler und andere datenabhängige Schritte ausschliesslich an den Trainingsdaten an.

# Cross-Validation mit Imputation

Bis hierher gab es **eine** feste Aufteilung (Training, Validation, Test). Cross-Validation (CV) wiederholt Training und Validierung mehrmals mit **wechselnden** Validierungsdaten. Die folgenden Kapitel zeigen Schritt für Schritt, wie das abläuft.

Grundregeln:

- Das **Testset bleibt ausserhalb** (IDs 9 und 10). Die CV läuft nur auf den Entwicklungsdaten, also Training + Validation (IDs 1 bis 8).
- Bei **k Folds** gibt es **k Durchläufe**. Jede Zeile ist genau **einmal** Validierung und **k-1 Mal** Training.
- In **jedem** Durchlauf wird alles neu gelernt: Imputer **und** Modell. Nichts wird von einem Durchlauf in den nächsten mitgenommen.

## 4. Datenaufteilung: Wer ist wann im Training?

Wir nehmen **k = 4 Folds**. Die 8 Entwicklungszeilen werden in 4 Teile zu je 2 Zeilen geteilt. Mit `shuffle=True` werden die Zeilen vorher zufällig gemischt (Seed 42), sonst wären die Folds feste Blöcke wie 1-2, 3-4 und so weiter.

In der Tabelle steht jede Zeile für eine ID, jede Spalte für einen Durchlauf:

- `Train`: in diesem Durchlauf zum Lernen verwendet
- `Val`: in diesem Durchlauf zum Bewerten verwendet
- `Test`: gehört nicht zur CV

In [ ]:
from sklearn.model_selection import KFold

X_dev = data.loc[1:8]  # Test (IDs 9, 10) stays out of the CV
cv = KFold(n_splits=4, shuffle=True, random_state=42)
folds = list(cv.split(X_dev))  # list of (train positions, validation positions)

plan = pd.DataFrame("Train", index=data.index, columns=[f"Durchlauf {i}" for i in range(1, 5)])
plan.loc[X_test.index, :] = "Test"
for i, (train_pos, val_pos) in enumerate(folds, start=1):
    plan.loc[X_dev.index[val_pos], f"Durchlauf {i}"] = "Val"

# every development row is validation exactly once
assert ((plan.loc[X_dev.index] == "Val").sum(axis=1) == 1).all()
plan

Lesehilfe: Pro Spalte sind genau 2 Zeilen `Val` und 6 Zeilen `Train`. Pro Zeile steht genau ein `Val` und drei `Train`. Die beiden Testzeilen sind in keinem Durchlauf dabei. Zusammen werden also alle 8 Entwicklungszeilen verwendet, aber nie alle gleichzeitig im selben Durchlauf.

## 5. Was passiert in einem Durchlauf mit der Imputation?

In jedem Durchlauf wird ein **neuer** Imputer nur auf den 6 Trainingszeilen dieses Durchlaufs angepasst. Danach füllt er die fehlenden Werte in Training und Validierungsfold mit diesen Medianen.

Die fehlenden Werte stecken in ID 2 (Einkommen) und ID 5 (Zimmer). Die Tabelle zeigt, ob diese Zeilen im Durchlauf Training oder Validierung sind und mit welchem Wert sie gefüllt werden.

In [ ]:
rows = []
for i, (train_pos, val_pos) in enumerate(folds, start=1):
    fold_imputer = SimpleImputer(strategy="median").fit(X_dev.iloc[train_pos])
    filled = pd.DataFrame(fold_imputer.transform(X_dev), columns=X_dev.columns, index=X_dev.index)
    rows.append({
        "Durchlauf": i,
        "Training-IDs": sorted(X_dev.index[train_pos]),
        "Validation-IDs": sorted(X_dev.index[val_pos]),
        "Median Einkommen": fold_imputer.statistics_[1],
        "ID 2 ist": plan.loc[2, f"Durchlauf {i}"],
        "ID 2 Einkommen wird": filled.loc[2, "Einkommen"],
        "Median Zimmer": fold_imputer.statistics_[2],
        "ID 5 ist": plan.loc[5, f"Durchlauf {i}"],
        "ID 5 Zimmer wird": filled.loc[5, "Zimmer"],
    })

pd.DataFrame(rows).set_index("Durchlauf")

Lesehilfe: Dieselbe Zeile (zum Beispiel ID 2) erhält je nach Durchlauf einen anderen Ersatzwert (65, 70, 80, 70), weil sich die Trainingszeilen und damit der Median ändern. Liegt ID 2 im Validierungsfold, hat sie den Median **nicht** mitbestimmt. Das ist der Sinn von «Imputer pro Durchlauf neu anpassen».

## 6. Alle Durchläufe: neues Modell, Score, verwerfen

Jetzt kommt das Modell dazu. Wir brauchen ein Ziel `y` und erfinden einen Preis pro ID. Der Ablauf pro Durchlauf:

1. Pipeline (Imputer + Ridge) **neu erzeugen** (`clone`)
2. Auf den Trainingszeilen `fit`: Imputer lernt Mediane, Modell lernt Koeffizienten
3. Auf den Validierungszeilen den Fehler (MAE) berechnen
4. Pipeline verwerfen

Zuerst als Schleife, damit jeder Schritt sichtbar ist. `cross_val_score` macht dasselbe in einer Zeile.

In [ ]:
from sklearn.base import clone
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline

y = pd.Series([210, 260, 330, 390, 450, 520, 580, 470, 300, 340], index=data.index, name="Preis")
y_dev, y_test = y.loc[1:8], y.loc[9:10]

pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", Ridge(alpha=1.0)),
])

fold_errors = []
for i, (train_pos, val_pos) in enumerate(folds, start=1):
    fold_pipeline = clone(pipeline)  # fresh imputer and fresh model
    fold_pipeline.fit(X_dev.iloc[train_pos], y_dev.iloc[train_pos])
    error = mean_absolute_error(y_dev.iloc[val_pos], fold_pipeline.predict(X_dev.iloc[val_pos]))
    fold_errors.append(error)
    print(f"Durchlauf {i}: trainiert auf {sorted(X_dev.index[train_pos])}, "
          f"geprüft auf {sorted(X_dev.index[val_pos])}, MAE {error:.1f}")

print(f"Mittelwert {pd.Series(fold_errors).mean():.1f}, Streuung {pd.Series(fold_errors).std(ddof=0):.1f}")

Dasselbe mit `cross_val_score`. Die Werte müssen übereinstimmen, weil dieselben Folds (`cv`) verwendet werden.

In [ ]:
scores = -cross_val_score(pipeline, X_dev, y_dev, cv=cv, scoring="neg_mean_absolute_error")
assert scores.round(6).tolist() == pd.Series(fold_errors).round(6).tolist()
print("MAE pro Durchlauf:", scores.round(1))

Die vier Fehler schwanken stark, weil jeder Fold nur 2 Validierungszeilen hat. Bei echten Daten sind die Folds grösser und die Streuung kleiner. Der **Mittelwert** bewertet das Setup (Imputer, Modelltyp, Hyperparameter), nicht ein einzelnes Modell: Alle vier Modelle werden danach verworfen.

## 7. Nach der CV: finales Modell und Test

Die CV dient zum **Auswählen und Bewerten**. Erst danach entsteht das Modell, das du wirklich verwendest:

1. Setup mit dem besten CV-Mittel wählen (zum Beispiel `alpha`, Median oder Mittelwert als Imputation). Bei n Kandidaten sind es n × k Trainings, zum Beispiel 5 × 4 = 20. `GridSearchCV` macht das automatisch.
2. Dieses Setup **einmal** auf allen Entwicklungsdaten (IDs 1 bis 8) trainieren. Der Imputer lernt dabei die Mediane aus allen 8 Zeilen.
3. **Einmal** auf dem Testset (IDs 9, 10) prüfen. Der Test wurde nie zum Auswählen benutzt.

In [ ]:
final_pipeline = clone(pipeline).fit(X_dev, y_dev)
test_mae = mean_absolute_error(y_test, final_pipeline.predict(X_test))
print(f"Testfehler (MAE): {test_mae:.1f}")